In [1]:
from tensorflow import keras
import tensorflow as tf
from tensorflow.keras.losses import CategoricalCrossentropy
from network import NetCNN1D


from mne.decoding import CSP
from sklearn.model_selection import StratifiedKFold

import numpy as np
import sys, os

import dataset_NewEEG
from config_NewEEG import Config

import matplotlib.pyplot as plt


import pandas as pd
from tabulate import tabulate


config = Config()



# Choose from: 'CLeft', 'CRight', 'CUp' and 'CDown'
config.used_classes = ['CLeft', 'CRight']
config.session_type = 'S2'


config.t_start = 0 
config.t_end = 3.0

window_size = 2.0
stride = 0.1


config.n_csp_components = 3

# All available channels = ['FC3', 'FCz', 'FC4', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'CP3', 'CPz', 'CP4']
config.used_channels = ['FC3', 'FCz', 'FC4', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'CP3', 'CPz', 'CP4']


config.start = int(config.t_start * 250)
config.t_end = int(config.t_end * 250)

In [2]:
def train_evaluate(session_ID, classes):

    config.used_classes = classes
    
    X, y = dataset_NewEEG.session_sliced_dataset(config ,session_ID, window_size, stride)

    crossValidation_KF = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


    acc = []

    shuffle_indices = np.random.permutation(len(y))


    X = X[shuffle_indices]
    y = y[shuffle_indices]

    for train_index, valid_index in crossValidation_KF.split(X,y):
        X_train = X[train_index]
        Y_train = y[train_index]
        X_valid = X[valid_index]
        Y_valid = y[valid_index]
        
        old_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')

        mu = X_train.mean()
        sigma = X_train.std( )

        #X_train = (X_train - mu) / sigma
        #X_valid =  (X_valid - mu) / sigma

        csp = CSP(n_components=config.n_csp_components, reg=None, log=None, norm_trace=False, transform_into='csp_space')

        X_train = csp.fit_transform(X_train, Y_train)
        X_valid = csp.transform(X_valid)


        X_train = X_train.transpose(0, 2, 1)
        X_valid = X_valid.transpose(0, 2, 1)

        Y_train = tf.keras.utils.to_categorical(Y_train, num_classes = 2)
        Y_valid = tf.keras.utils.to_categorical(Y_valid, num_classes = 2)



        model = NetCNN1D() 


        model.compile(
                loss= CategoricalCrossentropy(from_logits=True, label_smoothing=0.05),
                optimizer=keras.optimizers.Adam(learning_rate=1e-4, weight_decay = 2e-5), 
                metrics=[keras.metrics.CategoricalAccuracy(name='acc')]
        ) 



        sys.stdout.close()
        sys.stdout = old_stdout

        history = model.fit(X_train, Y_train, validation_data=(X_valid, Y_valid), epochs=100, batch_size=16, verbose=0)


        acc.append(max(history.history['val_acc']))



    print("================================== ")
    print(f"session {session_ID} = {100*np.mean(acc):.2f}% +- {100*np.std(acc):.1f}")

    return np.mean(acc)

In [3]:
classes_to_evaluate = [['CLeft', 'CRight'],
                       ['CUp', 'CDown'],
                       ['CUp', 'CRight']
                       ]


results_dict_S2 = [{} for i in range(len(classes_to_evaluate))]

for i in range(1, 12):
    session_ID = f'I{i:02d}'

    for c, classes in enumerate(classes_to_evaluate):    
        results_dict_S2[c][session_ID] = 100 * train_evaluate(session_ID, classes)



session I01 = 71.56% +- 5.5
session I01 = 87.61% +- 2.1
session I01 = 81.48% +- 2.1
session I02 = 79.67% +- 3.1
session I02 = 80.41% +- 3.4
session I02 = 84.11% +- 4.2
session I03 = 74.15% +- 3.3
session I03 = 82.19% +- 5.0
session I03 = 77.53% +- 1.9
session I04 = 93.68% +- 1.9
session I04 = 80.52% +- 3.2
session I04 = 85.68% +- 2.1
session I05 = 69.55% +- 3.5
session I05 = 84.48% +- 2.4
session I05 = 76.59% +- 5.1
session I06 = 83.40% +- 4.7
session I06 = 83.64% +- 1.8
session I06 = 80.45% +- 3.4
session I07 = 81.21% +- 1.5
session I07 = 64.97% +- 7.0
session I07 = 68.35% +- 4.5
session I08 = 79.97% +- 5.8
session I08 = 85.45% +- 1.2
session I08 = 82.09% +- 3.8
session I09 = 74.91% +- 4.9
session I09 = 71.27% +- 5.7
session I09 = 80.00% +- 3.0
session I10 = 82.08% +- 5.6
session I10 = 77.28% +- 3.2
session I10 = 74.55% +- 6.6
session I11 = 83.17% +- 4.4
session I11 = 77.40% +- 5.0
session I11 = 80.65% +- 2.5


In [4]:
pd.options.display.float_format = "{:.2f}".format

classes_to_evaluate = [str(c) for c in classes_to_evaluate]
df_S2 = pd.DataFrame(results_dict_S2, index=classes_to_evaluate)
df_S2.index.name = "Classes / Session_IDs"

df_S2["Row_Mean"] = df_S2.mean(axis=1)
df_S2.loc["Column_Mean"] = df_S2.mean(axis=0)

print(tabulate(df_S2, headers="keys", tablefmt="fancy_grid", floatfmt=".2f"))

╒═════════════════════════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤════════════╕
│ Classes / Session_IDs   │   I01 │   I02 │   I03 │   I04 │   I05 │   I06 │   I07 │   I08 │   I09 │   I10 │   I11 │   Row_Mean │
╞═════════════════════════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪════════════╡
│ ['CLeft', 'CRight']     │ 71.56 │ 79.67 │ 74.15 │ 93.68 │ 69.55 │ 83.40 │ 81.21 │ 79.97 │ 74.91 │ 82.08 │ 83.17 │      79.40 │
├─────────────────────────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼────────────┤
│ ['CUp', 'CDown']        │ 87.61 │ 80.41 │ 82.19 │ 80.52 │ 84.48 │ 83.64 │ 64.97 │ 85.45 │ 71.27 │ 77.28 │ 77.40 │      79.57 │
├─────────────────────────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼────────────┤
│ ['CUp', 'CRight']       │ 81.48 │ 84.11 │ 77.53 │ 85.68 │ 76.59 │ 80.45 │ 68.35 │ 82.09 │ 80.00

In [5]:
config.session_type = 'S3'


classes_to_evaluate = [['CLeft', 'CRight'],
                       ['CUp', 'CDown'],
                       ['CUp', 'CRight']
                       ]


results_dict_S3 = [{} for i in range(len(classes_to_evaluate))]

for i in range(1, 12):
    session_ID = f'I{i:02d}'

    for c, classes in enumerate(classes_to_evaluate):    
        results_dict_S3[c][session_ID] = 100 * train_evaluate(session_ID, classes)

session I01 = 80.00% +- 7.4
session I01 = 82.49% +- 6.5
session I01 = 77.10% +- 7.9
session I02 = 78.55% +- 6.0
session I02 = 65.05% +- 5.3
session I02 = 77.09% +- 4.7
session I03 = 76.36% +- 6.0
session I03 = 83.76% +- 4.3
session I03 = 73.23% +- 3.0
session I04 = 79.42% +- 6.1
session I04 = 81.21% +- 1.5
session I04 = 79.56% +- 4.8
session I05 = 78.33% +- 1.7
session I05 = 86.35% +- 3.4
session I05 = 79.13% +- 3.6
session I06 = 89.56% +- 10.1
session I06 = 94.92% +- 4.7
session I06 = 88.63% +- 6.1
session I07 = 82.75% +- 3.2
session I07 = 76.62% +- 2.8
session I07 = 84.98% +- 5.6
session I08 = 81.82% +- 6.6
session I08 = 82.83% +- 2.8
session I08 = 88.04% +- 4.2
session I09 = 92.03% +- 5.7
session I09 = 92.25% +- 4.8
session I09 = 83.01% +- 3.3
session I10 = 83.33% +- 4.7
session I10 = 77.52% +- 6.3
session I10 = 82.42% +- 4.7
session I11 = 83.03% +- 4.3
session I11 = 81.80% +- 6.5
session I11 = 85.73% +- 3.1


In [6]:
pd.options.display.float_format = "{:.2f}".format

classes_to_evaluate = [str(c) for c in classes_to_evaluate]
df_S3 = pd.DataFrame(results_dict_S3, index=classes_to_evaluate)
df_S3.index.name = "Classes / Session_IDs"

df_S3["Row_Mean"] = df_S3.mean(axis=1)
df_S3.loc["Column_Mean"] = df_S3.mean(axis=0)

print(tabulate(df_S3, headers="keys", tablefmt="fancy_grid", floatfmt=".2f"))

╒═════════════════════════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤════════════╕
│ Classes / Session_IDs   │   I01 │   I02 │   I03 │   I04 │   I05 │   I06 │   I07 │   I08 │   I09 │   I10 │   I11 │   Row_Mean │
╞═════════════════════════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪════════════╡
│ ['CLeft', 'CRight']     │ 80.00 │ 78.55 │ 76.36 │ 79.42 │ 78.33 │ 89.56 │ 82.75 │ 81.82 │ 92.03 │ 83.33 │ 83.03 │      82.29 │
├─────────────────────────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼────────────┤
│ ['CUp', 'CDown']        │ 82.49 │ 65.05 │ 83.76 │ 81.21 │ 86.35 │ 94.92 │ 76.62 │ 82.83 │ 92.25 │ 77.52 │ 81.80 │      82.25 │
├─────────────────────────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼────────────┤
│ ['CUp', 'CRight']       │ 77.10 │ 77.09 │ 73.23 │ 79.56 │ 79.13 │ 88.63 │ 84.98 │ 88.04 │ 83.01

## Session 2

In [7]:
print(tabulate(df_S2, headers="keys", tablefmt="fancy_grid", floatfmt=".2f"))

╒═════════════════════════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤════════════╕
│ Classes / Session_IDs   │   I01 │   I02 │   I03 │   I04 │   I05 │   I06 │   I07 │   I08 │   I09 │   I10 │   I11 │   Row_Mean │
╞═════════════════════════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪════════════╡
│ ['CLeft', 'CRight']     │ 71.56 │ 79.67 │ 74.15 │ 93.68 │ 69.55 │ 83.40 │ 81.21 │ 79.97 │ 74.91 │ 82.08 │ 83.17 │      79.40 │
├─────────────────────────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼────────────┤
│ ['CUp', 'CDown']        │ 87.61 │ 80.41 │ 82.19 │ 80.52 │ 84.48 │ 83.64 │ 64.97 │ 85.45 │ 71.27 │ 77.28 │ 77.40 │      79.57 │
├─────────────────────────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼────────────┤
│ ['CUp', 'CRight']       │ 81.48 │ 84.11 │ 77.53 │ 85.68 │ 76.59 │ 80.45 │ 68.35 │ 82.09 │ 80.00

## Session 3

In [8]:
print(tabulate(df_S3, headers="keys", tablefmt="fancy_grid", floatfmt=".2f"))

╒═════════════════════════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤═══════╤════════════╕
│ Classes / Session_IDs   │   I01 │   I02 │   I03 │   I04 │   I05 │   I06 │   I07 │   I08 │   I09 │   I10 │   I11 │   Row_Mean │
╞═════════════════════════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪════════════╡
│ ['CLeft', 'CRight']     │ 80.00 │ 78.55 │ 76.36 │ 79.42 │ 78.33 │ 89.56 │ 82.75 │ 81.82 │ 92.03 │ 83.33 │ 83.03 │      82.29 │
├─────────────────────────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼────────────┤
│ ['CUp', 'CDown']        │ 82.49 │ 65.05 │ 83.76 │ 81.21 │ 86.35 │ 94.92 │ 76.62 │ 82.83 │ 92.25 │ 77.52 │ 81.80 │      82.25 │
├─────────────────────────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼────────────┤
│ ['CUp', 'CRight']       │ 77.10 │ 77.09 │ 73.23 │ 79.56 │ 79.13 │ 88.63 │ 84.98 │ 88.04 │ 83.01